In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/shrutibhargava94/india-air-quality-data/data.csv', encoding='cp1252')

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df

In [ ]:
df.duplicated().sum()

In [ ]:
cleandf = df.drop_duplicates().copy()

In [ ]:
cleandf.duplicated().sum()

In [ ]:
cleandf

In [ ]:
cleandf.isnull().sum()

In [ ]:
cleandf['agency'] = df['agency'].fillna('Unknown')

In [ ]:
cleandf['agency'].isnull().sum()

In [ ]:
cleandf['type'] = (
    cleandf.groupby('location')['type']
      .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Unknown'))
)

In [ ]:
cleandf['type'].isnull().sum()

In [ ]:
cleandf = cleandf.dropna(
    subset=['so2', 'no2', 'rspm', 'spm', 'pm2_5'],
    how='all'
)

In [ ]:
cleandf.isnull().sum()

In [ ]:
cleandf['location'] = (
    cleandf.groupby('state')['location']
      .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else np.nan))
)

In [ ]:
cleandf['location'].isnull().sum()

In [ ]:
cleandf['location_monitoring_station'] = (
    cleandf.groupby('location')['location_monitoring_station']
      .transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else 'Unknown'))
)

In [ ]:
cleandf['location_monitoring_station'].isnull().sum()

In [ ]:
(cleandf['location_monitoring_station'] == 'Unknown').sum()

In [ ]:
cleandf['so2'] = (
    cleandf.groupby(['state','location'])['so2']
      .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
cleandf['so2'].isnull().sum()

In [ ]:
cleandf[cleandf['so2'].isnull()][['state', 'location']].drop_duplicates()

In [ ]:
cleandf[
    (cleandf['state'] == 'Madhya Pradesh') &
    (cleandf['location'] == 'Khajuraho')
]['so2']

In [ ]:
cleandf['no2'] = (
    cleandf.groupby(['state','location'])['no2']
      .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
cleandf['no2'].isnull().sum()

In [ ]:
cleandf['rspm'] = (
    cleandf.groupby(['state','location'])['rspm']
      .transform(lambda x: x.fillna(x.median()))
)

In [ ]:
cleandf['rspm'].isnull().sum()

In [ ]:
cleandf['date'] = pd.to_datetime(cleandf['date'])

mask = cleandf['date'].isna()

cleandf.loc[mask, 'date'] = pd.to_datetime(
    cleandf.loc[mask, 'sampling_date'],
    format='%B - M%m%Y',
    errors='coerce'
)

In [ ]:
cleandf['date'].isnull().sum()

In [ ]:
cleandf['date']

In [ ]:
cleandf[cleandf['date'].isnull()][['sampling_date', 'date']]

In [ ]:
cleandf = cleandf.dropna(subset=['date']).reset_index(drop=True)

In [ ]:
cleandf.isnull().sum()

In [ ]:
cleandf['location_monitoring_station']

In [ ]:
cleandf = cleandf.copy()

In [ ]:
cleandf.shape

In [ ]:
cleandf.info()

In [ ]:
cleandf['stn_code'].dropna().head(20)

In [ ]:
cleandf['stn_code'] = pd.to_numeric(
    cleandf['stn_code'],
    errors='coerce'
).astype('Int64')

In [ ]:
cleandf['stn_code'].dtype

In [ ]:
pd.to_numeric(
    cleandf['stn_code'],
    errors='coerce'
).dropna().mod(1).unique()

In [ ]:
cleandf[['stn_code']].head()

In [ ]:
cleandf[['stn_code']].isnull().sum()

In [ ]:
cleandf

In [ ]:
pollutant_cols = ['so2', 'no2', 'rspm', 'spm', 'pm2_5']

cleandf[pollutant_cols].describe()

In [ ]:
pollutant_cols = ['so2', 'no2', 'rspm', 'spm', 'pm2_5']

for col in pollutant_cols:
    Q1 = cleandf[col].quantile(0.25)
    Q3 = cleandf[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_mask = (
        (cleandf[col] < lower_bound) |
        (cleandf[col] > upper_bound)
    )
    
    outlier_count = outlier_mask.sum()
    non_null_count = cleandf[col].notna().sum()
    percentage = (outlier_count / non_null_count) * 100
    
    print(f"{col}")
    print(f"Lower bound : {lower_bound:.2f}")
    print(f"Upper bound : {upper_bound:.2f}")
    print(f"Outliers    : {outlier_count}")
    print(f"Percentage  : {percentage:.2f}%")
    print("-" * 40)

In [ ]:
for col in pollutant_cols:
    negative_count = (cleandf[col] < 0).sum()
    print(f"{col}: {negative_count} negative values")

In [ ]:
for col in pollutant_cols:
    zero_count = (cleandf[col] == 0).sum()
    print(f"{col}: {zero_count} zero values")

In [ ]:
cleandf.nlargest(10, 'so2')[
    ['state', 'location', 'type', 'so2', 'date']
]

In [ ]:
cleandf.nlargest(10, 'no2')[
    ['state', 'location', 'type', 'no2', 'date']
]

In [ ]:
cleandf.nlargest(10, 'rspm')[
    ['state', 'location', 'type', 'rspm', 'date']
]

In [ ]:
cleandf.nlargest(10, 'spm')[
    ['state', 'location', 'type', 'spm', 'date']
]

In [ ]:
cleandf.nlargest(10, 'pm2_5')[
    ['state', 'location', 'type', 'pm2_5', 'date']
]

In [ ]:
def so2_category(value):
    if pd.isna(value):
        return "Unknown"
    elif value <= 40:
        return "Low"
    elif value <= 80:
        return "Moderate"
    elif value <= 120:
        return "High"
    else:
        return "Very High"

cleandf['so2_category'] = cleandf['so2'].apply(so2_category)

In [ ]:
cleandf['so2_category'].value_counts()

In [ ]:
def no2_category(value):
    if pd.isna(value):
        return "Unknown"
    elif value <= 40:
        return "Low"
    elif value <= 80:
        return "Moderate"
    elif value <= 120:
        return "High"
    else:
        return "Very High"

cleandf['no2_category'] = cleandf['no2'].apply(no2_category)

In [ ]:
cleandf['no2_category'].value_counts()

In [ ]:
def pm25_category(value):
    if pd.isna(value):
        return "Unknown"
    elif value <= 30:
        return "Low"
    elif value <= 60:
        return "Moderate"
    elif value <= 90:
        return "High"
    else:
        return "Very High"

cleandf['pm25_category'] = cleandf['pm2_5'].apply(pm25_category)

In [ ]:
cleandf['pm25_category'].value_counts()

In [ ]:
cleandf[['so2_category', 'no2_category', 'pm25_category']].head()

In [ ]:
#feature engineering
cleandf['year'] = cleandf['date'].dt.year

In [ ]:
cleandf['year'].value_counts().sort_index()

In [ ]:
cleandf['month'] = cleandf['date'].dt.month

In [ ]:
cleandf['month_name'] = cleandf['date'].dt.month_name()

In [ ]:
cleandf['quarter'] = cleandf['date'].dt.quarter

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

cleandf['season'] = cleandf['month'].apply(get_season)

In [ ]:
cleandf['season'].value_counts()

In [ ]:
#decade group
cleandf['decade'] = (cleandf['year'] // 10) * 10
cleandf['decade'].value_counts().sort_index()

In [ ]:
pollutant_cols = ['so2', 'no2', 'rspm', 'spm', 'pm2_5']

cleandf['pollutants_available'] = (
    cleandf[pollutant_cols].notna().sum(axis=1)
)
cleandf['pollutants_available'].value_counts().sort_index()

In [ ]:
month_order = [
    'January', 'February', 'March', 'April',
    'May', 'June', 'July', 'August',
    'September', 'October', 'November', 'December'
]

cleandf['month_name'] = pd.Categorical(
    cleandf['month_name'],
    categories=month_order,
    ordered=True
)

In [ ]:
cleandf['month_name'].unique()

In [ ]:
cleandf[
    ['date', 'year', 'month', 'month_name',
     'quarter', 'season', 'decade',
     'pollutants_available']
].head(10)

In [ ]:
cleandf['year'] = cleandf['date'].dt.year
cleandf['month'] = cleandf['date'].dt.month
cleandf['month_name'] = cleandf['date'].dt.month_name()
cleandf['quarter'] = cleandf['date'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

cleandf['season'] = cleandf['month'].apply(get_season)

cleandf['decade'] = (cleandf['year'] // 10) * 10

pollutant_cols = ['so2', 'no2', 'rspm', 'spm', 'pm2_5']

cleandf['pollutants_available'] = (
    cleandf[pollutant_cols].notna().sum(axis=1)
)

In [ ]:
cleandf[
    ['date', 'year', 'month', 'month_name',
     'quarter', 'season', 'decade',
     'pollutants_available']
].head(10)

In [ ]:
cleandf['pollutants_available'].value_counts().sort_index()

In [ ]:
cleandf[['sampling_date', 'date']].head()

In [ ]:
pollutant_cols = ['so2', 'no2', 'rspm', 'spm', 'pm2_5']

correlation = cleandf[pollutant_cols].corr()

correlation

In [ ]:
cleandf[pollutant_cols].describe().T

In [ ]:
cleandf['state'].value_counts()

In [ ]:
cleandf['type'].value_counts()

In [ ]:
cleandf['agency'].value_counts().head(20)

In [ ]:
cleandf['location'].value_counts().head(20)

In [ ]:
analysis_df = cleandf[
    [
        'date',
        'year',
        'month',
        'month_name',
        'quarter',
        'season',
        'decade',
        'state',
        'location',
        'agency',
        'type',
        'so2',
        'no2',
        'rspm',
        'spm',
        'pm2_5',
        'so2_category',
        'no2_category',
        'pm25_category',
        'pollutants_available'
    ]
].copy()

In [ ]:
analysis_df.shape

In [ ]:
analysis_df.info()

In [ ]:
analysis_df = cleandf[
    [
        'date', 'year', 'month', 'month_name', 'quarter',
        'season', 'decade', 'state', 'location', 'agency',
        'type', 'so2', 'no2', 'rspm', 'spm', 'pm2_5',
        'so2_category', 'no2_category', 'pm25_category',
        'pollutants_available'
    ]
].copy()

In [ ]:
analysis_df.shape

In [ ]:
analysis_df[pollutant_cols].corr()

In [ ]:
analysis_df[pollutant_cols].describe().T

In [ ]:
analysis_df[['spm', 'pm2_5']].dropna().shape

In [ ]:
def available_pollutants(row):
    return ', '.join(
        col.upper().replace('PM2_5', 'PM2.5')
        for col in pollutant_cols
        if pd.notna(row[col])
    )

analysis_df['available_pollutants'] = analysis_df.apply(
    available_pollutants, axis=1
)

analysis_df['available_pollutants'].value_counts()

In [ ]:
analysis_df

In [ ]:
analysis_df.to_csv('preprocessed_data.csv', index= False)